# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.9 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

- Lecture 01: Tasks 3.1 and 3.2 use the string methods .strip() and .lower() (as pandas .str methods) so spellings like "Female " and " week12" match a single code, and .upper() to standardize site codes.
- Lecture 02: Tasks 3.1 and 3.2 use dictionaries as lookups, such as gender_map mapping each spelling to F, M, or X and visit_map mapping each visit label to baseline or week12.
- Lecture 03: After Task 4.3, complete_pairs (each patient's baseline and week-12 SBP, kept only when both exist) becomes a 27,149 × 2 NumPy array of patients × visits, and sbp_array.mean(axis=0) gives one mean SBP per visit: 143.01 mmHg at baseline and 137.15 mmHg at week 12. I used axis=0 because it collapses the rows (patients) to leave one mean per visit, whereas axis=1 would give 27,149 per-patient means and no axis would blend both visits into a single number.
- Lecture 04: Task 5.3 does a round trip, reading output/analysis_sample.csv back with dtype= and parse_dates= and confirming with round_trip.equals(analysis_sample) that the saved file holds the same labels, values, and dtypes, which works because Task 5.2 saved it with index=False.
- Lecture 05: Task 5.1 collects named validation rules in pd.Series({...}), such as unique (`patient_id`, `visit`) pairs and every recorded SBP passing .dropna().between(70, 250).all(), and assert checks.all(), checks[~checks]` stops the notebook before output/analysis_sample.csv can be saved if any rule fails.
- Lecture 06: Task 4.1 melts the wide A1c table into one row per patient and visit so it can join visits_clean, and Task 4.3 pivots SBP back to wide in output/sbp_wide_sample.csv, which works only because Task 3.2 already kept one reading per (`patient_id`, `visit`) pair.


## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
def profile(name, table):
    return {
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "exact_duplicate_rows": int(table.duplicated().sum()),
    }

results = []
for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

profile_dataframe = pd.DataFrame(results)
profile_dataframe.to_csv(PROFILE_PATH, index=False)
profile_dataframe

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

Grain and key of each cleaned table

| Table | Grain (one row = …) | Key |
|---|---|---|
| `patients.parquet` | one registered patient | `patient_id` |
| `sites.csv` (current records only) | one clinic site | `site_code` |
| `visits_north.csv` + `visits_south.csv` (stacked, later recheck kept) | one visit (baseline or week 12) by one patient | composite: (`patient_id`, `visit`) |
| `a1c_wide.parquet` (reshaped to long form) | one A1c result for one patient at one visit | composite: (`patient_id`, `visit`) |

The raw `a1c_wide.parquet` has a different grain: one row per patient, with one column per visit (`a1c_baseline`, `a1c_week12`). It already has one row per `patient_id`.

Raw rows that violate these keys

- Exact repeats: 45 rows in `patients.parquet` (a registry re-export) and 35 rows in each visit file (an export run twice) are identical to an earlier row. They are dropped.
- Visit rechecks: some visits have a second blood pressure reading taken 10 to 30 minutes after the first, with the same patient, visit, and date but a later `visit_time` and a different `sbp_mmhg`. The recheck can be listed before or after the first reading, and its `visit` label may be spelled differently. The reading with the later `visit_time` is kept.
- Current/retired site versions: `N01` and `S02` each appear twice in `sites.csv`, once with `record_status` "current" and once "retired". Only the current record is kept, leaving one row each for `N01`, `S02`, and `W03`.

Expected join cardinality (Task 4.4)

- Visits → patients: many-to-one (`validate="m:1"`). A patient has up to two visits; each visit belongs to one patient. Visits from patients not in the registry keep their row with missing patient values.
- Patients → sites: many-to-one (`validate="m:1"`). Many patients share one site; each site code has one current record. Patients whose site code has no current site (such as `E04`) keep their row with missing site values.
- Visits → long-form A1c results: one-to-one (`validate="1:1"`). Each visit has at most one A1c result and each result belongs to one visit. Lab results for patients with no visits are not kept by the left merge.

### 2.2 Stack the two visit files

In [4]:
north = raw["visits_north"].copy(deep=True)
north["source_file"] = "visits_north.csv"

south = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})
south["source_file"] = "visits_south.csv"

visits_raw = pd.concat([north, south], ignore_index=True)

print(visits_raw.shape)
visits_raw.head()

(57371, 7)


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


### 2.3 Audit the raw tables

In [5]:
patients = raw["patients"]
sites = raw["sites"]

gender = patients["gender"]
gender_empty = gender.str.strip() == ""
gender_not_coded = ~gender_empty & ~gender.isin(["F", "M", "X"])

sentinel_year = patients["birth_year"] == "9999"

site = patients["site_code"]
site_needs_fix = site.str.upper().str.replace(r"[\s-]", "", regex=True) != site

site_repeated = sites.duplicated(subset=["site_code"], keep=False)

visit = visits_raw["visit"]
visit_not_labeled = ~visit.isin(["baseline", "week12"])

sbp = pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce")
sbp_sentinel = sbp == 999
sbp_outside = sbp.notna() & (sbp != 999) & ((sbp < 70) | (sbp > 250))

date_text = visits_raw["visit_date"]
date_shape_ok = date_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
date_parsed = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
date_fail = ~(date_shape_ok & date_parsed.notna())

visit_key = visit.str.strip().str.lower().str.replace(" ", "", regex=False)
visit_mapped = visit_key.map({"baseline": "baseline", "bl": "baseline",
                              "week12": "week12", "wk12": "week12"})
pair_repeated = (visits_raw.assign(visit_mapped=visit_mapped)
                 .duplicated(subset=["patient_id", "visit_mapped"], keep=False))

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs",
    ],
    "count": [int(mask.sum()) for mask in [
        gender_not_coded, gender_empty,sentinel_year, site_needs_fix, site_repeated,
        visit_not_labeled, sbp_sentinel, sbp_outside, date_fail, pair_repeated,
    ]],
})
issue_audit.to_csv(AUDIT_PATH, index=False)
issue_audit
assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

## Task 3: Clean

### 3.1 Clean the patients

In [6]:
patients_clean = raw["patients"].copy(deep=True)
print(patients_clean["gender"].value_counts(dropna=False), "\n")

patients_clean = patients_clean.drop_duplicates(keep="first")

gender_map = {
    "f": "F", "female": "F", "woman": "F",
    "m": "M", "male": "M", "man": "M",
    "x": "X", "nonbinary": "X", "non-binary": "X", "nb": "X",
}
patients_clean["gender"] = (patients_clean["gender"].str.strip().str.lower()
                            .map(gender_map).astype("category"))

year = pd.to_numeric(patients_clean["birth_year"]).astype("Int64")
patients_clean["birth_year"] = year.mask(year == 9999)

patients_clean["site_code"] = (patients_clean["site_code"].str.upper()
                               .str.replace("-", "", regex=False)
                               .str.replace(" ", "", regex=False))

patients_clean = patients_clean.reset_index(drop=True)
patients_clean.head()

gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow] 



,patient_id,gender,birth_year,site_code
0,P00001,F,1966,S02
1,P00002,F,<NA>,N01
2,P00003,F,1970,N01
3,P00004,X,<NA>,N01
4,P00005,M,1947,N01


In [7]:
print(patients_clean["gender"].cat.categories)

summary = pd.DataFrame({
    "metric": [
        "patients", "birth_year missing", "birth_year min", "birth_year max",
        "site_code values", "patients at N01", "patients at S02", "patients at E04",
    ],
    "value": [
        len(patients_clean),
        int(patients_clean["birth_year"].isna().sum()),
        int(patients_clean["birth_year"].min()),
        int(patients_clean["birth_year"].max()),
        patients_clean["site_code"].nunique(),
        int((patients_clean["site_code"] == "N01").sum()),
        int((patients_clean["site_code"] == "S02").sum()),
        int((patients_clean["site_code"] == "E04").sum()),
    ],
})
summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)
print(summary.to_string(index=False), "\n")

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)
gender_counts

Index(['F', 'M', 'X'], dtype='str')
            metric  value
          patients  30000
birth_year missing    371
    birth_year min   1941
    birth_year max   1996
  site_code values      3
   patients at N01  13739
   patients at S02  15706
   patients at E04    555 



,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [8]:
visits_clean = visits_raw.copy(deep=True)
print(visits_clean["visit"].value_counts(dropna=False), "\n")

visits_clean = visits_clean.drop_duplicates(keep="first")

visit_map = {
    "baseline": "baseline", "bl": "baseline",
    "week12": "week12", "week 12": "week12", "wk12": "week12",
}
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(visit_map)

sbp = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
in_range = sbp.between(70, 250).fillna(False).astype(bool)
visits_clean["sbp_mmhg"] = sbp.where(in_range).astype("Int64")

visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce").astype("float64")

date_text = visits_clean["visit_date"]
exact_shape = date_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False).astype(bool)
parsed = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
visits_clean["visit_date"] = parsed.where(exact_shape)

visits_clean.head()

visit
baseline     28533
week12       26529
BASELINE       256
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        195
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow] 



,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,week12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


In [9]:
pair_columns= ["patient_id", "visit"]
repeats = visits_clean[visits_clean.duplicated(subset=pair_columns, keep=False)]
print(len(repeats), "rows in repeated pairs")
display(repeats.sort_values(pair_columns + ["visit_time"]).head(10))

visits_clean = (visits_clean.sort_values("visit_time", kind="stable")
                .drop_duplicates(subset=pair_columns, keep="last"))

240 rows in repeated pairs


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
85,P00088,baseline,2026-03-10,09:50,152,68.2,visits_north.csv
26491,P00183,week12,2026-05-25,14:15,134,78.9,visits_south.csv
26490,P00183,week12,2026-05-25,14:35,133,78.9,visits_south.csv
26684,P00377,baseline,2026-01-30,09:05,147,101.0,visits_south.csv
26685,P00377,baseline,2026-01-30,09:15,139,101.0,visits_south.csv


In [10]:
visits_clean = (visits_clean.sort_values(["patient_id", "visit"])
                .reset_index(drop=True)
                [["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]])

print(visits_clean["visit"].value_counts(dropna=False), "\n")

summary = pd.DataFrame({
    "metric": [
        "visits", "baseline visits", "week12 visits",
        "visit_date missing", "visit_date min", "visit_date max",
        "sbp_mmhg missing", "sbp_mmhg min", "sbp_mmhg max",
        "weight_kg missing", "weight_kg mean",
        "visits from visits_north.csv", "visits from visits_south.csv",
    ],
    "value": [
        len(visits_clean),
        int((visits_clean["visit"] == "baseline").sum()),
        int((visits_clean["visit"] == "week12").sum()),
        int(visits_clean["visit_date"].isna().sum()),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        int(visits_clean["sbp_mmhg"].isna().sum()),
        int(visits_clean["sbp_mmhg"].min()),
        int(visits_clean["sbp_mmhg"].max()),
        int(visits_clean["weight_kg"].isna().sum()),
        round(visits_clean["weight_kg"].mean(), 2),
        int((visits_clean["source_file"] == "visits_north.csv").sum()),
        int((visits_clean["source_file"] == "visits_south.csv").sum()),
    ],
})
summary.to_csv(VISITS_SUMMARY_PATH, index=False)
summary

visit
baseline    29599
week12      27582
Name: count, dtype: int64 



,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [11]:
decision_table = pd.DataFrame(
    [
        ["patients", "exact duplicate rows",
         "I dropped exact duplicates, keeping the first copy.",
         "The 45 repeats came from a registry re-export and provided no new information."],
        ["patients", "gender spellings that were not a code",
         "I stripped and lowercased each spelling, mapped it to F, M, or X, and made unknown or empty values missing.",
         "The staff typed the same genders many ways, and unknown or blank entries carried no gender information."],
        ["patients", "site codes needing normalization",
         "Uppercased site codes and removed hyphens and spaces.",
         "Variants like N-01 and n01 are the same site and must match sites.csv for the join."],
        ["patients", "birth_year sentinel 9999",
         "I replaced 9999 with missing and stored birth_year as Int64.",
         "9999 is a placeholder for an unknown year, so treating it as a real year would distort ages."],
        ["visits", "weight column named wt_kg in visits_south.csv",
         "Renamed wt_kg to weight_kg before stacking the two files.",
         "Both columns hold weight in kg, and matching names keep them in one column when stacked."],
        ["visits", "sbp sentinel 999 and readings outside 70 through 250",
         "Made 999 and any reading outside 70 through 250 missing and stored sbp_mmhg as Int64.",
         "999 marks a missing reading, and values outside that range are not plausible blood pressures."],
        ["visits", "impossible and loosely written visit dates",
         "Kept only exact YYYY-MM-DD dates that exist on the calendar and made the rest missing.",
         "Dates like 2026-02-30 cannot be real, and loosely written dates could be misread."],
        ["visits", "repeated patient_id and visit pairs",
         "Dropped exact duplicates, then kept the later reading of each recheck by visit_time.",
         "A recheck is a second reading taken minutes later, so the later one is the final measurement."],
        ["sites", "repeated site_code rows",
         "Keep only the current record for each site_code.",
         "Retired records hold old site names, and keeping both would duplicate patients in the join."],
    ],
    columns=["table", "issue", "decision", "reason"],
)
decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,"I dropped exact duplicates, keeping the first ...",The 45 repeats came from a registry re-export ...
1,patients,gender spellings that were not a code,"I stripped and lowercased each spelling, mappe...","The staff typed the same genders many ways, an..."
2,patients,site codes needing normalization,Uppercased site codes and removed hyphens and ...,Variants like N-01 and n01 are the same site a...
3,patients,birth_year sentinel 9999,I replaced 9999 with missing and stored birth_...,"9999 is a placeholder for an unknown year, so ..."
4,visits,weight column named wt_kg in visits_south.csv,Renamed wt_kg to weight_kg before stacking the...,"Both columns hold weight in kg, and matching n..."
5,visits,sbp sentinel 999 and readings outside 70 throu...,Made 999 and any reading outside 70 through 25...,"999 marks a missing reading, and values outsid..."
6,visits,impossible and loosely written visit dates,Kept only exact YYYY-MM-DD dates that exist on...,"Dates like 2026-02-30 cannot be real, and loos..."
7,visits,repeated patient_id and visit pairs,"Dropped exact duplicates, then kept the later ...",A recheck is a second reading taken minutes la...
8,sites,repeated site_code rows,Keep only the current record for each site_code.,"Retired records hold old site names, and keepi..."


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
a1c_long = raw["a1c_wide"].copy(deep=True).melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct",
)

a1c_long["visit"] = a1c_long["visit"].map({"a1c_baseline": "baseline", "a1c_week12": "week12"})

assert a1c_long["a1c_pct"].dtype == "float64", a1c_long["a1c_pct"].dtype
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

visits_order = ["baseline", "week12"]
a1c_summary = pd.DataFrame({
    "visit": visits_order,
    "results": [int((a1c_long["visit"] == v).sum()) for v in visits_order],
    "a1c_pct_mean": [round(a1c_long.loc[a1c_long["visit"] == v, "a1c_pct"].mean(), 2)
                     for v in visits_order],
})
a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)
a1c_summary

,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [13]:
sites = raw["sites"]

try:
    patients_clean.merge(sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error, "\n")

display(sites[sites.duplicated(subset=["site_code"], keep=False)])

sites_current = (sites.loc[sites["record_status"] == "current", ["site_code", "site_name"]]
                 .reset_index(drop=True))
sites_current

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ... 



,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
1,S02,South Valley Clinic
2,W03,West Clinic


### 4.3 Pivot blood pressure to one row per patient

In [14]:
sbp_wide = visits_clean.pivot(index="patient_id", columns="visit", values="sbp_mmhg").reset_index()
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]

sbp_wide_sample = sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)]
sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index=False)
sbp_wide_sample

complete_pairs = sbp_wide[["baseline", "week12"]].dropna()
sbp_array = np.array(complete_pairs, dtype=float)

print("shape:", sbp_array.shape)
print("mean SBP per visit (baseline, week12):", sbp_array.mean(axis=0).round(2))

shape: (27149, 2)
mean SBP per visit (baseline, week12): [143.01 137.15]


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

### 4.4 Join the analysis table

In [15]:
analysis = (
    visits_clean
    .merge(patients_clean, on="patient_id", how="left", validate="many_to_one")
    .merge(sites_current, on="site_code", how="left", validate="many_to_one")
    .merge(a1c_long, on=["patient_id", "visit"], how="left", validate="one_to_one")
)
assert len(analysis) == len(visits_clean), "A merge changed the number of visit rows."

columns = [
    "patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "a1c_pct",
    "gender", "birth_year", "site_code", "site_name", "source_file",
]
analysis = analysis[columns].astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",
})

print(analysis.shape)
print(analysis.dtypes)
analysis.head()

(57181, 11)
patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00002,baseline,2026-02-20,132,85.2,8.4,F,<NA>,N01,North Clinic,visits_north.csv
3,P00002,week12,2026-05-08,129,82.0,7.4,F,<NA>,N01,North Clinic,visits_north.csv
4,P00003,baseline,2026-01-16,148,67.5,5.9,F,1970,N01,North Clinic,visits_north.csv


## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [16]:
sbp_present = analysis["sbp_mmhg"].dropna()
site_names_present = analysis["site_name"].dropna()

checks = pd.Series({
    "Analysis rows equal visits_clean rows": len(analysis) == len(visits_clean),
    "no repeated patient_id and visit pair": not analysis.duplicated(subset=["patient_id", "visit"]).any(),
    "sbp_mmhg present values within 70 through 250": bool(sbp_present.between(70, 250).all()),
    "site_name present values are current sites": bool(site_names_present.isin(sites_current["site_name"]).all()),
    "sbp_wide patient_id is unique": bool(sbp_wide["patient_id"].is_unique),
    "Raw tables equal their snapshots": all(raw[name].equals(raw_snapshot[name]) for name in raw),
})
display(checks)

assert checks.all(), checks[~checks]

Analysis rows equal visits_clean rows            True
no repeated patient_id and visit pair            True
sbp_mmhg present values within 70 through 250    True
site_name present values are current sites       True
sbp_wide patient_id is unique                    True
Raw tables equal their snapshots                 True
dtype: bool

### 5.2 Save the analysis extract

In [17]:
analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)

assert len(analysis_sample) == visits_clean["patient_id"].isin(SAMPLE_IDS).sum()
print(len(analysis_sample), "rows")
analysis_sample

28 rows


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [18]:
read_dtypes = {
    "patient_id": "string",
    "visit": "string",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",
}
round_trip = pd.read_csv(ANALYSIS_SAMPLE_PATH, dtype=read_dtypes, parse_dates=["visit_date"])

print(round_trip.equals(analysis_sample))

True
